In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(''))

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = os.path.join('..', 'data', 'fully combined')

qb_raw = pd.read_csv(os.path.join(DATA_DIR, 'qb_master.csv'))
rb_raw = pd.read_csv(os.path.join(DATA_DIR, 'rb_master.csv'))
te_raw = pd.read_csv(os.path.join(DATA_DIR, 'te_master.csv'))
wr_raw = pd.read_csv(os.path.join(DATA_DIR, 'wr_all_seasons_without_playoffs.csv'))

for name, df in [('QB', qb_raw), ('RB', rb_raw), ('TE', te_raw), ('WR', wr_raw)]:
    scol = 'Season' if 'Season' in df.columns else 'season'
    print(f'{name}: {df.shape[0]} redova x {df.shape[1]} kolona | Sezone: {df[scol].min()}-{df[scol].max()}')


In [ ]:
# QB: GS>0 filter (backup QB-ovi se uklanjaju - odluka iz EDA)
qb = qb_raw[qb_raw['GS'] > 0].copy()
qb['target'] = qb['Yds'] / qb['G'].replace(0, np.nan)

# RB: rushing yards per game
rb = rb_raw.copy()
rb['target'] = rb['Rush_Yds'] / rb['G'].replace(0, np.nan)

# TE: G>0 filter (igrači koji nisu odigrali ni jednu utakmicu — povreda, G=0 → target=NaN)
te = te_raw[te_raw['G'] > 0].copy()
te['target'] = te['Rec_Yds'] / te['G']

# WR: log transformacija na receiving_yards (odluka iz EDA - power-law distribucija)
# Clip na 0 pre log1p jer negativni yds/game su artefakti (penali)
wr = wr_raw.copy()
wr['rec_yds_per_game'] = wr['receiving_yards'] / wr['games_played'].replace(0, np.nan)
wr['target'] = np.log1p(wr['rec_yds_per_game'].clip(lower=0))

# Pregled
print('Ciljne promenljive:')
for name, df in [('QB', qb), ('RB', rb), ('TE', te)]:
    t = df['target']
    print(f'{name} (Yds/G):       mean={t.mean():.2f}, median={t.median():.2f}, min={t.min():.2f}, max={t.max():.2f}')

t_raw = wr['rec_yds_per_game']
t_log = wr['target']
print(f'WR (Yds/G raw):   mean={t_raw.mean():.2f}, median={t_raw.median():.2f}, min={t_raw.min():.2f}, max={t_raw.max():.2f}')
print(f'WR (log1p Yds/G): mean={t_log.mean():.3f}, median={t_log.median():.3f}, min={t_log.min():.3f}, max={t_log.max():.3f}')


In [ ]:
# Award encoding — 5 binarnih kolona (odluka iz EDA)
# Kategorije: PB, AP-1, AP-2, MVP top5 (1-5), OPoY top5 (1-5)
# Isključujemo: Comeback Player of Year, rangiranja van top5

def encode_awards(series):
    def has(val, keywords):
        if pd.isna(val) or str(val).strip() == '':
            return 0
        return int(any(kw in str(val) for kw in keywords))
    
    df_enc = pd.DataFrame(index=series.index)
    df_enc['award_PB']       = series.apply(lambda v: has(v, ['PB']))
    df_enc['award_AP1']      = series.apply(lambda v: has(v, ['AP-1']))
    df_enc['award_AP2']      = series.apply(lambda v: has(v, ['AP-2']))
    df_enc['award_MVP_top5'] = series.apply(lambda v: has(v, ['MVP-1','MVP-2','MVP-3','MVP-4','MVP-5']))
    df_enc['award_OPoY_top5']= series.apply(lambda v: has(v, ['OPoY-1','OPoY-2','OPoY-3','OPoY-4','OPoY-5']))
    return df_enc

# QB i TE imaju 'Awards' (uppercase), RB ima 'awards' (lowercase)
qb = pd.concat([qb, encode_awards(qb['Awards'])], axis=1)
rb = pd.concat([rb, encode_awards(rb['awards'])], axis=1)
te = pd.concat([te, encode_awards(te['Awards'])], axis=1)
# WR nema awards kolonu — preskačemo

award_cols = ['award_PB','award_AP1','award_AP2','award_MVP_top5','award_OPoY_top5']
print('Award encoding — broj sezona po kategoriji:')
print(f'\n{"Nagrada":<20} {"QB":>6} {"RB":>6} {"TE":>6}')
print('-' * 38)
for col in award_cols:
    print(f'{col:<20} {qb[col].sum():>6} {rb[col].sum():>6} {te[col].sum():>6}')


In [ ]:
# ── Uklanjanje nepotrebnih kolona ─────────────────────────────────────────

# QB: metapodaci, defensive stats, elo (puno null-ova), special teams, duplikati
QB_DROP = (
    ['PlayerID', 'Pos', 'QBrec', 'Lng', 'TD%', 'Int%', 'Sk%', 'Y/G']  # metapodaci + duplikati
    + [c for c in qb.columns if c.startswith('def_')]                   # defensive stats
    + [c for c in qb.columns if c.startswith('elo_')]                   # elo kolone
    + ['snp_ST%', 'snp_special_teams']                                  # special teams snaps
)

# RB: metapodaci, defensive stats, special teams, duplikati kolona
RB_DROP = (
    ['PlayerID', 'Lg', 'Pos', 'Rush_Y/G', 'Rec_Y/G']                   # metapodaci + duplikati
    + [c for c in rb.columns if c.startswith('def_')]                   # defensive stats
    + ['snp_ST%', 'snp_ST_Snaps', 'snp_special_teams']                 # special teams
    + ['rec_success', 'catch_pct', 'Y/Tgt', 'Touch',                   # duplikati
       'yds_per_touch', 'yds_from_scrimmage', 'rush_receive_td', 'Rec_R/G']
)

# TE: metapodaci, special teams, defensive snaps
TE_DROP = (
    ['PlayerID', 'Pos']
    + ['snp_ST%', 'snp_ST_Snaps', 'snp_Def%', 'snp_Def_Snaps']
)

# WR: ID-evi, duplikati tima, quarter splits, win-prob splits, weather, game-state, betting
WR_DROP = (
    ['receiver_player_id', 'passer_player_id',                          # ID-evi
     'defteam', 'home_team', 'away_team', 'player_team']                # duplikati tima
    + [c for c in wr.columns if '_Q1' in c or '_Q2' in c
       or '_Q3' in c or '_Q4' in c]                                     # quarter splits
    + [c for c in wr.columns if c.startswith('yards_wp_')
       or c.startswith('receptions_wp_') or c.startswith('targets_wp_')]# win-prob splits
    + ['temp_f', 'humidity_pct', 'wind_mph',                            # weather
       'is_rain', 'is_snow', 'is_clear', 'is_dome', 'surface']         # stadium
    + ['avg_score_diff', 'avg_quarter', 'trailing_pct',                 # game-state
       'leading_pct', 'wp_var']
    + ['pregame_spread', 'pregame_total']                               # betting linije
)

# Primena — drop samo kolona koje postoje (ignorišemo ako neka već nedostaje)
qb.drop(columns=[c for c in QB_DROP if c in qb.columns], inplace=True)
rb.drop(columns=[c for c in RB_DROP if c in rb.columns], inplace=True)
te.drop(columns=[c for c in TE_DROP if c in te.columns], inplace=True)
wr.drop(columns=[c for c in WR_DROP if c in wr.columns], inplace=True)

# Statistika posle brisanja
print(f'{"Pozicija":<6} {"Pre":>6} {"Posle":>6} {"Izbačeno":>9}')
print('-' * 30)
for pos, raw, df in [('QB', qb_raw, qb), ('RB', rb_raw, rb), ('TE', te_raw, te), ('WR', wr_raw, wr)]:
    print(f'{pos:<6} {raw.shape[1]:>6} {df.shape[1]:>6} {raw.shape[1] - df.shape[1]:>9}')

print('\nPreostale kolone po poziciji:')
for pos, df in [('QB', qb), ('RB', rb), ('TE', te), ('WR', wr)]:
    print(f'\n{pos}: {list(df.columns)}')


In [ ]:
# Analiza QB adv_ kolona: null% i korelacija sa osnovnim statistikama

adv_cols = [c for c in qb.columns if c.startswith('adv_')]
print(f'Ukupno adv_ kolona: {len(adv_cols)}\n')

# 1. Null procenat po koloni
null_pct = qb[adv_cols].isnull().mean() * 100
print('Null% po adv_ koloni:')
print(null_pct.sort_values(ascending=False).to_string())

# 2. Korelacija adv_ kolona sa osnovnim statistikama (Yds, Cmp, Att, TD, Rate itd.)
basic_stats = ['Yds', 'Cmp', 'Att', 'Cmp%', 'TD', 'Int', 'Rate', 'Sk', 'AV']
print('\n\nKorelacija adv_ kolona sa osnovnim statistikama (|r| > 0.85 = visoka duplikacija):')
print(f'{"adv kolona":<45} {"max |r|":>8}  {"najvise korelirana osnovna"}')
print('-' * 85)
for col in adv_cols:
    valid = qb[[col] + basic_stats].dropna()
    if len(valid) < 30:
        print(f'{col:<45} {"(premalo podataka)":>8}')
        continue
    corrs = valid[basic_stats].corrwith(valid[col]).abs()
    max_r = corrs.max()
    max_col = corrs.idxmax()
    marker = ' ◄ DUPLIKAT?' if max_r > 0.85 else ''
    print(f'{col:<45} {max_r:>8.3f}  {max_col}{marker}')


In [ ]:
# Izbacujemo sve adv_ kolone iz QB (67-100% null-ovi, duplikati osnovnih statistika)
adv_cols_to_drop = [c for c in qb.columns if c.startswith('adv_')]
qb.drop(columns=adv_cols_to_drop, inplace=True)

print(f'Izbačeno {len(adv_cols_to_drop)} adv_ kolona.')
print(f'QB shape: {qb.shape}')
print(f'Preostale kolone: {list(qb.columns)}')


In [ ]:
for pos, df in [('QB', qb), ('RB', rb), ('TE', te), ('WR', wr)]:
    print(f'\n{"="*60}')
    print(f'  {pos} — {df.shape[1]} kolona x {df.shape[0]} redova')
    print(f'{"="*60}')
    for i, col in enumerate(df.columns, 1):
        print(f'  {i:>3}. {col}')


In [ ]:
# QB: izbacujemo rr_* i snp_* kolone
qb_extra_drop = (
    [c for c in qb.columns if c.startswith('rr_')]
    + ['snp_Def%', 'snp_Off%', 'snp_defense', 'snp_offense']
)
qb.drop(columns=[c for c in qb_extra_drop if c in qb.columns], inplace=True)

# RB, TE, WR: izbacujemo adv_* i snp_* kolone
for df in [rb, te, wr]:
    drop = [c for c in df.columns if c.startswith('adv_') or c.startswith('snp_')]
    df.drop(columns=drop, inplace=True)

print(f'{"Pozicija":<6} {"Kolona":>7} {"Redova":>7}')
print('-' * 25)
for pos, df in [('QB', qb), ('RB', rb), ('TE', te), ('WR', wr)]:
    print(f'{pos:<6} {df.shape[1]:>7} {df.shape[0]:>7}')


In [ ]:
# Dodatno čišćenje kolona

# Awards raw string — više ne treba (već enkodovano u award_* binarnim kolonama)
for df, col in [(qb, 'Awards'), (rb, 'awards'), (te, 'Awards')]:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

# QB: QBR — 33.9% null, strukturalni (postoji tek od ~2006), izbacujemo
if 'QBR' in qb.columns:
    qb.drop(columns=['QBR'], inplace=True)

# RB: receiving stats + kombinovane kolone sa null-ovima (rec komponenta nedostaje)
RB_DROP2 = ['Tgt', 'Rec', 'Rec_Yds', 'Rec_Y/R', 'Rec_TD', 'Rec_1D',
            'Rec_Succ%', 'Rec_Lng', 'Rec/G', 'Catch%', 'Rec_Y/Tgt', 'rec_long',
            'Y/Touch', 'Touches', 'Scrimmage_Yds', 'Rush_Rec_TD']
rb.drop(columns=[c for c in RB_DROP2 if c in rb.columns], inplace=True)

# TE: rushing stats (TE model predviđa receiving, rush stats su sekundarne)
TE_DROP2 = ['Rush_Att', 'Rush_Yds', 'Rush_TD', 'Rush_1D', 'Rush_Y/G', 'Rush_A/G',
            'Touches', 'Y/Touch', 'Scrimmage_Yds', 'Rush_Succ%', 'Rush_Lng', 'Rush_Y/A']
te.drop(columns=[c for c in TE_DROP2 if c in te.columns], inplace=True)

# WR: defensive deviation stats
WR_DROP2 = ['def_targets_dev', 'def_receptions_dev', 'def_yards_dev',
            'def_tds_dev', 'def_epa_dev']
wr.drop(columns=[c for c in WR_DROP2 if c in wr.columns], inplace=True)

print(f'{"Pozicija":<6} {"Kolona":>7} {"Redova":>7}')
print('-' * 25)
for pos, df in [('QB', qb), ('RB', rb), ('TE', te), ('WR', wr)]:
    print(f'{pos:<6} {df.shape[1]:>7} {df.shape[0]:>7}')


In [ ]:

for pos, df in [('QB', qb), ('RB', rb), ('TE', te), ('WR', wr)]:
    print(f'\n{"="*60}')
    print(f'  {pos} — {df.shape[1]} kolona x {df.shape[0]} redova')
    print(f'{"="*60}')
    for i, col in enumerate(df.columns, 1):
        print(f'  {i:>3}. {col}')


In [ ]:
# Analiza null vrednosti po poziciji
for pos, df in [('QB', qb), ('RB', rb), ('TE', te), ('WR', wr)]:
    null_counts = df.isnull().sum()
    has_nulls = null_counts[null_counts > 0]
    print(f'\n{"="*55}')
    print(f'  {pos} — kolone sa null vrednostima ({len(has_nulls)} od {df.shape[1]})')
    print(f'{"="*55}')
    if len(has_nulls) == 0:
        print('  Nema null vrednosti.')
    else:
        print(f'  {"Kolona":<35} {"Null#":>6}  {"Null%":>6}')
        print(f'  {"-"*52}')
        for col, cnt in has_nulls.sort_values(ascending=False).items():
            pct = cnt / len(df) * 100
            print(f'  {col:<35} {cnt:>6}  {pct:>5.1f}%')


In [ ]:

# ── Kreiranje lag feature matrice (t-1 i t-2) ─────────────────────────────
# Model predviđa performansu sezone t koristeći SAMO podatke poznate prije sezone t:
#   - Sve statistike iz t-1 (_lag1) i t-2 (_lag2)
#   - Age i Team_Changed iz t (znamo ih prije sezone)
#   - target = Yds/G (ili log1p za WR) sezone t
#
# KONSEKUTIVNOST SEZONA [Provjera 2]:
#   Koristimo merge na (player, season) umjesto groupby.shift().
#   lag1 redovi dobijaju season+1, pa left join pronalazi TAČNO sezonu t-1.
#   Ako igrač nema podatke za t-1 (rookie, propuštena sezona), merge vraća NaN.
#   → Nema false pozitiva gdje bi starija sezona bila tretirana kao t-1.
#
# IMPUTACIJA POLITIKA [Promjena 6]:
#   lag2 NaN → 0  (nema historije 2 sezone unazad — neutralna vrijednost)
#   lag1 NaN → NaN (imputiraće se medijanom u koraku imputacije, ne nulom)

POS_CONFIG = {
    'QB': dict(df=qb, player_col='Player', season_col='Season',
               static_feats=['Age', 'Team_Changed'],
               id_cols=['Player', 'Season', 'Team']),
    'RB': dict(df=rb, player_col='Player', season_col='Season',
               static_feats=['Age', 'Team_Changed'],
               id_cols=['Player', 'Season', 'Team']),
    'TE': dict(df=te, player_col='Player', season_col='Season',
               static_feats=['Age', 'Team_Changed'],
               id_cols=['Player', 'Season', 'Team']),
    'WR': dict(df=wr, player_col='receiver_player_name', season_col='season',
               static_feats=['age', 'team_changed'],
               id_cols=['receiver_player_name', 'season', 'posteam']),
}

lagged = {}
for pos, cfg in POS_CONFIG.items():
    df       = cfg['df'].copy()
    pcol     = cfg['player_col']
    scol     = cfg['season_col']
    id_cols  = cfg['id_cols']
    static   = [c for c in cfg['static_feats'] if c in df.columns]

    # Kolone koje lagujemo: sve osim id-eva, statičkih feature-a i targeta
    lag_cols = [c for c in df.columns
                if c not in id_cols + static + ['target']]

    # Baza: id + statički feature-i + target (sve iz sezone t)
    base = df[id_cols + static + ['target']].copy()

    # Helper DF samo za lag join
    lag_src = df[[pcol, scol] + lag_cols].copy()

    # Lag 1 — pomjeramo sezonu za +1 da matchamo sa tekućom sezonom t
    lag1 = lag_src.rename(columns={c: f'{c}_lag1' for c in lag_cols}).copy()
    lag1[scol] = lag1[scol] + 1
    base = base.merge(lag1, on=[pcol, scol], how='left')

    # Lag 2 — pomjeramo sezonu za +2
    lag2 = lag_src.rename(columns={c: f'{c}_lag2' for c in lag_cols}).copy()
    lag2[scol] = lag2[scol] + 2
    base = base.merge(lag2, on=[pcol, scol], how='left')

    lag1_cols = [f'{c}_lag1' for c in lag_cols]
    lag2_cols = [f'{c}_lag2' for c in lag_cols]

    # lag2 → 0 (nema historije 2 sezone unazad)
    # lag1 → ostaju NaN → imputiraće se medijanom u koraku imputacije
    base[lag2_cols] = base[lag2_cols].fillna(0)

    # Ukloni redove bez targeta (edge cases G=0)
    base = base.dropna(subset=['target']).reset_index(drop=True)

    lagged[pos] = base

# Provjera — prikaz lag1 NaN redova (igrači bez prethodne sezone)
print(f'{"Pos":<5} {"Redova":>8} {"Kolona":>8} {"lag1 feats":>12} {"lag2 feats":>12} {"lag1 NaN redovi":>16}')
print('-' * 65)
for pos, df in lagged.items():
    lag1_cols = [c for c in df.columns if c.endswith('_lag1')]
    lag2_cols = [c for c in df.columns if c.endswith('_lag2')]
    lag1_nan_rows = df[lag1_cols].isnull().any(axis=1).sum()
    print(f'{pos:<5} {df.shape[0]:>8} {df.shape[1]:>8} {len(lag1_cols):>12} {len(lag2_cols):>12} {lag1_nan_rows:>16}')


In [ ]:

# Train/test split — temporalni: Season < 2024 = train, Season == 2024 = test
# Radi na lag feature matrici (lagged dict) — nema curenja podataka iz sezone t

POS_SPLIT = {
    'QB': dict(id_cols=['Player', 'Season', 'Team'],                    season_col='Season'),
    'RB': dict(id_cols=['Player', 'Season', 'Team'],                    season_col='Season'),
    'TE': dict(id_cols=['Player', 'Season', 'Team'],                    season_col='Season'),
    'WR': dict(id_cols=['receiver_player_name', 'season', 'posteam'],   season_col='season'),
}

splits = {}
for pos, cfg in POS_SPLIT.items():
    df       = lagged[pos]
    id_cols  = cfg['id_cols']
    scol     = cfg['season_col']

    train_df = df[df[scol] < 2024].copy()
    test_df  = df[df[scol] == 2024].copy()

    feat_cols = [c for c in df.columns if c not in id_cols + ['target']]

    X_train = train_df[feat_cols]
    y_train = train_df['target']
    X_test  = test_df[feat_cols]
    y_test  = test_df['target']

    splits[pos] = (X_train, X_test, y_train, y_test)

print(f'{"Pos":<5} {"X_train":>10} {"X_test":>8} {"y_train":>9} {"y_test":>8} {"features":>10}')
print('-' * 55)
for pos, (X_tr, X_te, y_tr, y_te) in splits.items():
    print(f'{pos:<5} {X_tr.shape[0]:>10} {X_te.shape[0]:>8} {y_tr.shape[0]:>9} {y_te.shape[0]:>8} {X_tr.shape[1]:>10}')


In [ ]:

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Binary kolone — ne skaliramo (već su 0/1, interpretabilnost se čuva)
# Award kolone sada imaju _lag1 / _lag2 sufiks (jer su lagovane)
# Team_Changed / team_changed ostaju statički (iz sezone t) → bez sufiksa
BINARY_COLS = [
    'award_PB_lag1',       'award_AP1_lag1',       'award_AP2_lag1',
    'award_MVP_top5_lag1', 'award_OPoY_top5_lag1',
    'award_PB_lag2',       'award_AP1_lag2',       'award_AP2_lag2',
    'award_MVP_top5_lag2', 'award_OPoY_top5_lag2',
    'Team_Changed', 'team_changed',
]

processed = {}

for pos, (X_train, X_test, y_train, y_test) in splits.items():

    # ── 1. Imputacija — fit samo na train ────────────────────────
    # PROMJENA 6: lag2 NaN → 0 (već urađeno u lag creation, ali eksplicitno ovdje)
    #             ostali NaN (lag1 za rookies) → MEDIJAN iz train seta
    #             Medijan je robusniji od mean-a na outlierima.

    lag2_present = [c for c in X_train.columns if c.endswith('_lag2')]
    X_train = X_train.copy()
    X_test  = X_test.copy()
    X_train[lag2_present] = X_train[lag2_present].fillna(0)
    X_test[lag2_present]  = X_test[lag2_present].fillna(0)

    # Medijanska imputacija za preostale null-ove (lag1 za rookies)
    imputer = SimpleImputer(strategy='median')
    X_train_imp = pd.DataFrame(
        imputer.fit_transform(X_train),
        columns=X_train.columns, index=X_train.index
    )
    X_test_imp = pd.DataFrame(
        imputer.transform(X_test),
        columns=X_test.columns, index=X_test.index
    )

    # ── 2. Standardizacija — samo numeričke (ne binary) kolone ───
    binary_present = [c for c in BINARY_COLS if c in X_train.columns]
    scale_cols     = [c for c in X_train.columns if c not in binary_present]

    scaler = StandardScaler()
    X_train_imp[scale_cols] = scaler.fit_transform(X_train_imp[scale_cols])
    X_test_imp[scale_cols]  = scaler.transform(X_test_imp[scale_cols])

    processed[pos] = (X_train_imp, X_test_imp, y_train, y_test)

# Provjera — nullovi nakon imputacije
print(f'{"Pos":<5} {"X_train nulls":>14} {"X_test nulls":>13} {"scaled cols":>12} {"binary cols":>12}')
print('-' * 60)
for pos, (X_tr, X_te, _, _) in processed.items():
    binary_present = [c for c in BINARY_COLS if c in X_tr.columns]
    scale_cols     = [c for c in X_tr.columns if c not in binary_present]
    print(f'{pos:<5} {X_tr.isnull().sum().sum():>14} {X_te.isnull().sum().sum():>13} '
          f'{len(scale_cols):>12} {len(binary_present):>12}')


In [ ]:

# Pregled kolona i null vrijednosti u processed datasetu

for pos, (X_tr, X_te, y_tr, y_te) in processed.items():
    print(f'\n{"="*65}')
    print(f'  {pos} — X_train: {X_tr.shape[0]} redova x {X_tr.shape[1]} kolona')
    print(f'{"="*65}')
    
    # Sve kolone sa null statusom i tipom
    null_counts = X_tr.isnull().sum()
    print(f'\n  {"#":>4}  {"Kolona":<45} {"Tip":>8}  {"Nulls":>6}')
    print(f'  {"-"*70}')
    for i, col in enumerate(X_tr.columns, 1):
        null_n = null_counts[col]
        marker = ' ◄ NULL!' if null_n > 0 else ''
        print(f'  {i:>4}. {col:<45} {str(X_tr[col].dtype):>8}  {null_n:>6}{marker}')
    
    total_nulls = null_counts.sum()
    print(f'\n  Ukupno null-ova u X_train: {total_nulls}')
    print(f'  Ukupno null-ova u X_test:  {X_te.isnull().sum().sum()}')
    print(f'  y_train nulls: {y_tr.isnull().sum()} | y_test nulls: {y_te.isnull().sum()}')


In [ ]:

# ── GridSearchCV — optimizacija hiperparametara + cross-validacija ─────────
# PROMJENA 1: Umjesto fiksnih hiperparametara, koristimo GridSearchCV sa
# TimeSeriesSplit(n=5) za svaki model. Best estimator se koristi za test eval.
#
# Scoring: neg_mean_absolute_error (za WR — na log skali, ali relativo poređenje
# modela ostaje korektno; WR inverzija se radi pri test izveštaju).
#
# NAPOMENA za WR: GridSearchCV bira najbolje parametre u log1p prostoru.
# Ovo je prihvatljivo jer je redosljed modela po kvalitetu konzistentan
# između log i linear skale.

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np, pandas as pd, copy

MODEL_CONFIGS = {
    'LinearRegression': {
        'model': LinearRegression(),
        'params': {},
    },
    'Ridge': {
        'model': Ridge(),
        'params': {'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]},
    },
    'Lasso': {
        'model': Lasso(max_iter=10000),
        'params': {'alpha': [0.001, 0.01, 0.1, 1.0]},
    },
    'ElasticNet': {
        'model': ElasticNet(max_iter=10000),
        'params': {'alpha': [0.01, 0.1, 1.0], 'l1_ratio': [0.1, 0.5, 0.9]},
    },
    'KNeighbors': {
        'model': KNeighborsRegressor(),
        'params': {'n_neighbors': [3, 5, 7, 9, 11, 15], 'weights': ['uniform', 'distance']},
    },
    'RandomForest': {
        'model': RandomForestRegressor(random_state=42, n_jobs=-1),
        'params': {
            'n_estimators': [100, 200, 300],
            'max_depth': [5, 10, None],
            'min_samples_split': [2, 5],
        },
    },
}

tscv = TimeSeriesSplit(n_splits=5)

best_estimators = {}   # {pos: {model_name: fitted_best_estimator}}
cv_results      = {}   # {pos: DataFrame(Model, CV_MAE, Best_params)}

for pos, (X_tr, X_te, y_tr, y_te) in processed.items():
    best_estimators[pos] = {}
    rows = []
    log_note = ' [log skala]' if pos == 'WR' else ''

    print(f'\n{"="*68}')
    print(f'  {pos} — GridSearchCV (n_train={len(X_tr)}){log_note}')
    print(f'{"="*68}')
    print(f'  {"Model":<20} {"CV MAE":>8}  {"Najbolji parametri"}')
    print(f'  {"-"*65}')

    for name, cfg in MODEL_CONFIGS.items():
        if cfg['params']:
            grid = GridSearchCV(
                copy.deepcopy(cfg['model']),
                cfg['params'],
                cv=tscv,
                scoring='neg_mean_absolute_error',
                n_jobs=-1,
                refit=True,
            )
            grid.fit(X_tr, y_tr)
            best_model = grid.best_estimator_
            cv_mae     = -grid.best_score_
            best_params = grid.best_params_
        else:
            best_model = copy.deepcopy(cfg['model'])
            best_model.fit(X_tr, y_tr)
            cv_mae = -cross_val_score(
                copy.deepcopy(cfg['model']), X_tr, y_tr,
                cv=tscv, scoring='neg_mean_absolute_error', n_jobs=-1
            ).mean()
            best_params = {}

        best_estimators[pos][name] = best_model
        rows.append({'Model': name, 'CV_MAE': round(cv_mae, 3), 'Best_params': str(best_params)})
        params_str = str(best_params) if best_params else '-'
        print(f'  {name:<20} {cv_mae:>8.3f}  {params_str}')

    cv_results[pos] = pd.DataFrame(rows).set_index('Model')

print('\nGridSearchCV završen za sve pozicije.')


In [ ]:

# ── Test evaluacija na sezoni 2024 ─────────────────────────────────────────
# Koristimo best_estimators iz GridSearchCV (refit=True → već fitovani na
# punom X_train). Za WR: target je log1p(yds/g) → inverzno transformišemo
# za SVE metrike (MAE, RMSE, R²) → direktno usporedivo sa QB/RB/TE.

test_results = {}   # {pos: DataFrame sa test metrikama}
best_models  = {}   # {pos: (ime_modela, model_objekt)}

for pos, (X_tr, X_te, y_tr, y_te) in processed.items():
    rows = []
    is_wr = (pos == 'WR')

    print(f'\n{"="*60}')
    print(f'  {pos} — Test evaluacija (sezona 2024, n_test={len(X_te)})')
    if is_wr:
        print(f'  [WR: sve metrike na originalnoj yds/g skali (expm1)]')
    print(f'{"="*60}')
    print(f'  {"Model":<20} {"MAE":>8} {"RMSE":>8} {"R²":>8}  {"Best params"}')
    print(f'  {"-"*75}')

    for name, model in best_estimators[pos].items():
        preds = model.predict(X_te)

        if is_wr:
            # Inverz log1p → yds/g skala za sve metrike
            y_true_inv = np.expm1(y_te)
            y_pred_inv = np.expm1(preds)
            mae  = mean_absolute_error(y_true_inv, y_pred_inv)
            rmse = np.sqrt(mean_squared_error(y_true_inv, y_pred_inv))
            r2   = r2_score(y_true_inv, y_pred_inv)   # R² na yds/g skali
        else:
            mae  = mean_absolute_error(y_te, preds)
            rmse = np.sqrt(mean_squared_error(y_te, preds))
            r2   = r2_score(y_te, preds)

        rows.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
        best_params_str = cv_results[pos].loc[name, 'Best_params']
        print(f'  {name:<20} {mae:>8.3f} {rmse:>8.3f} {r2:>8.3f}  {best_params_str}')

    df_test = pd.DataFrame(rows).set_index('Model')
    test_results[pos] = df_test

    best_name = df_test['RMSE'].idxmin()
    best_models[pos] = (best_name, best_estimators[pos][best_name])
    print(f'\n  ★ Najbolji model ({pos}): {best_name} | RMSE={df_test.loc[best_name,"RMSE"]:.3f}')

print('\nFinalna test evaluacija završena.')


In [ ]:

# ── Sumarni prikaz GridSearchCV MAE + Test metrike + čuvanje best modela ──
import joblib, os

MODELS_DIR = os.path.join('..', 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

print('GridSearchCV CV MAE vs Test rezultati\n')
print(f'{"Pos":<5} {"Model":<20} {"CV_MAE":>8} │ {"Test_MAE":>9} {"Test_RMSE":>10} {"Test_R²":>8}')
print('─' * 70)

for pos in ['QB', 'RB', 'TE', 'WR']:
    cv_df   = cv_results[pos]
    test_df = test_results[pos]
    best    = best_models[pos][0]

    for i, model_name in enumerate(MODEL_CONFIGS.keys()):
        if model_name not in test_df.index:
            continue
        marker = ' ★' if model_name == best else ''
        prefix = pos if i == 0 else ''
        cv_mae   = cv_df.loc[model_name, 'CV_MAE']
        test_row = test_df.loc[model_name]
        print(f'{prefix:<5} {model_name:<20} '
              f'{cv_mae:>8.3f} │ '
              f'{test_row["MAE"]:>9.3f} {test_row["RMSE"]:>10.3f} {test_row["R2"]:>8.3f}{marker}')
    print('─' * 70)

# Čuvanje best modela po poziciji
print('\nSačuvani modeli:')
for pos, (name, model) in best_models.items():
    path = os.path.join(MODELS_DIR, f'{pos}_best_model.joblib')
    joblib.dump(model, path)
    test_rmse = test_results[pos].loc[name, 'RMSE']
    best_params = cv_results[pos].loc[name, 'Best_params']
    print(f'  {pos}: {name} → RMSE={test_rmse:.3f} | params={best_params}')


In [ ]:

# ── Predicted vs Actual — scatter plotovi za sve pozicije ─────────────────
# WR: predikcije i stvarne vrijednosti se konvertuju expm1() → yds/g skala.
# Sve 4 pozicije su tako direktno usporedive na istoj skali (yds/g).

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

COLORS = {'QB': '#3498db', 'RB': '#e74c3c', 'TE': '#9b59b6', 'WR': '#2ecc71'}

# Dohvati imena igrača iz test seta (sezona 2024)
PLAYER_COL = {
    'QB': 'Player', 'RB': 'Player', 'TE': 'Player', 'WR': 'receiver_player_name'
}
SEASON_COL = {
    'QB': 'Season', 'RB': 'Season', 'TE': 'Season', 'WR': 'season'
}

player_names = {}
for pos in ['QB', 'RB', 'TE', 'WR']:
    pcol = PLAYER_COL[pos]
    scol = SEASON_COL[pos]
    test_df = lagged[pos][lagged[pos][scol] == 2024]
    player_names[pos] = test_df[pcol].values

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

for ax, pos in zip(axes.flat, ['QB', 'RB', 'TE', 'WR']):
    _, X_te, _, y_te = processed[pos]
    best_name, best_model = best_models[pos]
    preds = best_model.predict(X_te)
    names = player_names[pos]

    if pos == 'WR':
        y_actual = np.expm1(y_te.values)
        y_pred   = np.expm1(preds)
        scale_note = ' [expm1]'
    else:
        y_actual = y_te.values
        y_pred   = preds
        scale_note = ''

    mae  = test_results[pos].loc[best_name, 'MAE']
    rmse = test_results[pos].loc[best_name, 'RMSE']
    r2   = test_results[pos].loc[best_name, 'R2']

    ax.scatter(y_actual, y_pred, color=COLORS[pos], alpha=0.55, s=35,
               edgecolor='white', linewidth=0.4)

    lo = min(y_actual.min(), y_pred.min()) * 0.95
    hi = max(y_actual.max(), y_pred.max()) * 1.05
    ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.45, linewidth=1.2, label='Idealno')

    # Dodaj imena igrača na tačke
    for x, y, name in zip(y_actual, y_pred, names):
        # Skrati ime: prvo slovo + prezime (npr. "P. Mahomes")
        parts = str(name).split()
        short = f'{parts[0][0]}. {" ".join(parts[1:])}' if len(parts) > 1 else name
        ax.annotate(
            short,
            xy=(x, y),
            xytext=(3, 3),
            textcoords='offset points',
            fontsize=6.5,
            alpha=0.75,
            color='#222222',
        )

    ax.set_xlabel('Stvarne vrijednosti (yds/g)', fontsize=11)
    ax.set_ylabel('Predviđene vrijednosti (yds/g)', fontsize=11)
    ax.set_title(
        f'{pos} — {best_name}{scale_note}\n'
        f'MAE={mae:.1f}  RMSE={rmse:.1f}  R²={r2:.3f}',
        fontsize=12, fontweight='bold'
    )
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)

fig.suptitle(
    'Predicted vs Actual — Najbolji model po poziciji (Test sezona 2024)\n'
    'WR: yds/g skala (expm1 inverz log1p)',
    fontsize=14, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.show()
